<a href="https://colab.research.google.com/github/fatimaibadovva-a11y/matrix_academy/blob/main/smart_waiter_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
import os
import re

from langchain_text_splitters import MarkdownHeaderTextSplitter

import warnings

warnings.filterwarnings("ignore")

In [ ]:
with open("narxana_restoran.md", "r", encoding="utf-8") as f:
    markdown_text = f.read()

print(markdown_text[:2000])

# Narxana Restoranlar Şəbəkəsi

> **Qeyd:** *Bu sənəd tədris məqsədilə hazırlanmış uydurma restoran haqqındadır. Bütün adlar, ünvanlar, telefon nömrələri, şəxslər və tərəfdaş şirkətlər uydurmadır.*

## Ümumi məlumat

**Narxana** 2009-cu ildə Bakıda açılmış Azərbaycan və Qafqaz mətbəxi restoranlar şəbəkəsidir. Restoranın adı *"nar"* və *"xana"* (ev, məkan) sözlərindən yaranıb və *"narın evi"* mənasını verir. Nar Azərbaycan mədəniyyətində bolluğun, bərəkətin və süfrə mədəniyyətinin simvoludur. Buna görə də narı həm loqomuzda, həm də menyumuzun bir çox yeməyində görə bilərsiniz.

Hazırda şəbəkənin 6 şəhərdə 9 filialı, 320-dən çox əməkdaşı var və hər gün orta hesabla 4 500 qonağa xidmət göstərir. Narxananın əsas prinsipi belədir: ***"Nənə reseptləri, müasir təqdimat."***

- **Mətbəx növü:** Azərbaycan, Qafqaz, qismən Aralıq dənizi mətbəxi
- **Qiymət səviyyəsi:** orta (bir nəfər üçün orta hesab 25–40 AZN)
- **Halal:** Bütün ət məhsulları halal sertifikatlıdır
- **Alkoqol:** Yalnız İçərişəhə

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

chunks = markdown_splitter.split_text(markdown_text)

print("Number of chunks:", len(chunks))

Number of chunks: 196


In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
processed_chunks = []

def count_tokens(text):
    return len(
        embedding_model.tokenizer.encode(
            text,
            add_special_tokens=True
        )
    )

def split_long_line(line, prefix, max_tokens=128):
    words = line.split()
    parts = []
    current_words = []

    for word in words:
        candidate = " ".join(current_words + [word])

        if count_tokens(prefix + candidate) <= max_tokens:
            current_words.append(word)
        else:
            if current_words:
                parts.append(" ".join(current_words))
            current_words = [word]

    if current_words:
        parts.append(" ".join(current_words))

    return parts


for chunk in chunks:
    metadata = chunk.metadata

    path = " > ".join(
        str(value)
        for value in metadata.values()
        if value
    )

    prefix = f"Path: {path}\n\n"

    lines = [
        line for line in chunk.page_content.splitlines()
        if line.strip()
    ]

    current_lines = []

    for line in lines:

        # Əgər tək sətir özü 128 token-dən uzundursa
        if count_tokens(prefix + line) > 128:

            if current_lines:
                processed_chunks.append(
                    prefix + "\n".join(current_lines)
                )
                current_lines = []

            long_line_parts = split_long_line(
                line,
                prefix,
                max_tokens=128
            )

            for part in long_line_parts:
                processed_chunks.append(prefix + part)

        else:
            candidate = "\n".join(current_lines + [line])

            if count_tokens(prefix + candidate) <= 128:
                current_lines.append(line)

            else:
                if current_lines:
                    processed_chunks.append(
                        prefix + "\n".join(current_lines)
                    )

                current_lines = [line]

    if current_lines:
        processed_chunks.append(
            prefix + "\n".join(current_lines)
        )

print("Total processed chunks:", len(processed_chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (170 > 128). Running this sequence through the model will result in indexing errors


Total processed chunks: 264


In [ ]:
token_lengths = [
    count_tokens(chunk)
    for chunk in processed_chunks
]

print("Number of chunks:", len(processed_chunks))
print("Maximum token length:", max(token_lengths))

Number of chunks: 264
Maximum token length: 128


In [ ]:
embeddings = embedding_model.encode(
    processed_chunks,
    show_progress_bar=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Number of embeddings: 264
Embedding dimension: 384


In [ ]:
!pip install -U faiss-cpu

In [ ]:
import faiss
import numpy as np

In [ ]:
embedding_matrix = np.array(embeddings).astype("float32")

print("Shape:", embedding_matrix.shape)

Shape: (264, 384)


In [ ]:
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 264


In [ ]:
menu_chunks = []

for chunk in processed_chunks:
    if "Menyu" in chunk:
        menu_chunks.append(chunk)

print("Total menu chunks:", len(menu_chunks))

Total menu chunks: 194


In [ ]:
menu_embeddings = embedding_model.encode(
    menu_chunks,
    show_progress_bar=True
)

print("Number of menu embeddings:", len(menu_embeddings))
print("Embedding dimension:", len(menu_embeddings[0]))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Number of menu embeddings: 194
Embedding dimension: 384


In [ ]:
menu_embedding_matrix = np.array(menu_embeddings).astype("float32")

menu_dimension = menu_embedding_matrix.shape[1]

menu_index = faiss.IndexFlatL2(menu_dimension)
menu_index.add(menu_embedding_matrix)

print("Number of vectors in menu index:", menu_index.ntotal)

Number of vectors in menu index: 194


In [ ]:
def menu_retriever(query, k=5):
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    distances, indices = menu_index.search(
        query_embedding,
        k
    )

    results = []

    query_lower = query.lower()

    # Əsas kateqoriyaları önə çək
    category_keywords = [
        "əsas yeməklər",
        "plovlar",
        "şorbalar",
        "salatlar",
        "desertlər",
        "səhər yeməkləri",
        "imza yeməkləri"
    ]

    for chunk in menu_chunks:
        chunk_lower = chunk.lower()

        for category in category_keywords:
            if category.split()[0] in query_lower and category in chunk_lower:
                if chunk not in results:
                    results.append(chunk)

    # Xüsusi etiketləri önə çək
    tag_keywords = [
        "qlütensiz",
        "vegan",
        "vegetarian",
        "imza yeməyi"
    ]

    for chunk in menu_chunks:
        chunk_lower = chunk.lower()

        for tag in tag_keywords:
            if tag in query_lower and tag in chunk_lower:
                if chunk not in results:
                    results.append(chunk)

    # FAISS semantic nəticələri
    for idx in indices[0]:
        chunk = menu_chunks[idx]

        if chunk not in results:
            results.append(chunk)

    return results[:k]

In [ ]:
!pip install -U openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [ ]:
CHAT_MODEL = "openrouter/free"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Chat model:", CHAT_MODEL)
print("Embedding model:", EMBEDDING_MODEL)

Chat model: openrouter/free
Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [ ]:
!pip install -U diffusers transformers accelerate safetensors

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

image_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
)

image_pipe = image_pipe.to("cuda")

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionXLPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionXLPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' obje

In [ ]:
image_cache = {}

In [ ]:
def generate_image(prompt_en):
    cache_key = prompt_en.strip().lower()

    if cache_key in image_cache:
        print("Image loaded from cache.")
        image = image_cache[cache_key]
        display(image)
        return image

    image = image_pipe(
        prompt=prompt_en,
        num_inference_steps=1,
        guidance_scale=0.0
    ).images[0]

    image_cache[cache_key] = image

    display(image)

    return image

In [ ]:
def create_image_prompt(user_prompt):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Convert the user's request into a short English prompt "
                    "for food image generation. Return only the English prompt."
                )
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    prompt_en = response.choices[0].message.content.strip()

    return prompt_en

In [ ]:
def handle_image_request(user_prompt):
    prompt_en = create_image_prompt(user_prompt)

    print("English image prompt:", prompt_en)

    image = generate_image(prompt_en)

    return image

In [ ]:
def rag_retriever(query, k=5):
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    # 1. Query ilə Path arasında uyğunluq
    query_lower = query.lower()

    for chunk in processed_chunks:
        path = chunk.split("\n")[0].lower()

        path_parts = [
            part.strip()
            for part in path.split(">")
        ]

        for part in path_parts:
            words = part.split()

            if any(
                len(word) >= 5 and word in query_lower
                for word in words
            ):
                if chunk not in results:
                    results.append(chunk)
                break

    # 2. FAISS semantic nəticələri
    for idx in indices[0]:
        chunk = processed_chunks[idx]

        if chunk not in results:
            results.append(chunk)

    return results[:k]

In [ ]:
SYSTEM_PROMPT = """
Sən Narxana Restoranlar Şəbəkəsinin ağıllı ofisiant botusan.

Qaydalar:
1. İstifadəçi başqa dil istəməsə, Azərbaycan dilində cavab ver.
2. Restoran haqqında faktları yalnız verilmiş CONTEXT-dən götür.
3. Qiymət, faiz, vaxt, porsiya sayı və digər rəqəmləri uydurma.
4. Rəqəmləri yalnız CONTEXT-də olan məlumatlardan istifadə edərək de.
5. Menyu suallarında yalnız menyu məlumatlarına əsaslan.
6. Cavab CONTEXT-də yoxdursa, məlumatın tapılmadığını açıq şəkildə bildir.
7. Cavabları qısa, aydın və nəzakətli ver.
8. Cavabı yalnız Azərbaycan dilində yaz.
9. Türk, ingilis, ispan və ya başqa dillərdən sözlər qarışdırma.
10. CONTEXT-dəki faktları dəyişdirmə və olmayan məlumat əlavə etmə.
"""

In [ ]:
def answer_question(user_prompt):
    prompt_lower = user_prompt.lower()

    # Image generation
    if "generate" in prompt_lower:
        prompt_en = create_image_prompt(user_prompt)
        generate_image(prompt_en)

        return "Şəkil generasiya edildi."

    # Menu RAG
    if "menu" in prompt_lower:
        retrieved_chunks = menu_retriever(
            user_prompt,
            k=5
        )

    # General RAG
    else:
        retrieved_chunks = rag_retriever(
            user_prompt,
            k=5
        )

    context = "\n\n".join(retrieved_chunks)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
CONTEXT:
{context}

SUAL:
{user_prompt}
"""
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
def start_chat():
    history = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    print("Narxana ağıllı ofisiant botuna xoş gəlmisiniz!")
    print("Menyu sualları üçün 'menu' sözündən istifadə edin.")
    print("Şəkil yaratmaq üçün 'generate' sözündən istifadə edin.")
    print("Çıxmaq üçün 'exit' və ya 'quit' yazın.\n")

    while True:
        user_prompt = input("Siz: ").strip()

        # Exit
        if user_prompt.lower() in ["exit", "quit"]:
            print("Bot: Sağ olun! Yenidən gözləyirik.")
            break

        prompt_lower = user_prompt.lower()

        # Image generation
        if "generate" in prompt_lower:
            prompt_en = create_image_prompt(user_prompt)
            generate_image(prompt_en)

            answer = "Şəkil generasiya edildi."

        else:
            # Menu RAG
            if "menu" in prompt_lower:
                retrieved_chunks = menu_retriever(
                    user_prompt,
                    k=5
                )

            # General RAG
            else:
                retrieved_chunks = rag_retriever(
                    user_prompt,
                    k=5
                )

            context = "\n\n".join(retrieved_chunks)

            messages = history + [
                {
                    "role": "user",
                    "content": f"""
CONTEXT:
{context}

SUAL:
{user_prompt}
"""
                }
            ]

            response = client.chat.completions.create(
                model=CHAT_MODEL,
                messages=messages
            )

            answer = response.choices[0].message.content

        print("Bot:", answer)

        # History
        history.append(
            {
                "role": "user",
                "content": user_prompt
            }
        )

        history.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        # System prompt + son 5 dialoq cütü
        history = [history[0]] + history[-10:]

In [ ]:
test_questions = [
    "Narxana neçə şəhərdə fəaliyyət göstərir?",
    "İçərişəhər filialında spirtli içki varmı?",
    "Restoranda rezervasiya necə edilir?",
    "Çatdırılmada sifariş 90 dəqiqədən çox gecikərsə nə baş verir?",
    "Nar Xalı proqramında Qızıl səviyyənin cashback faizi neçədir?",
    "menu Düşbərənin qiyməti və tərkibi nədir?",
    "menu Qlütensiz əsas yemək tövsiyə et.",
    "menu 3-4 nəfərlik imza yeməyi tövsiyə et.",
    "Restorana ev heyvanı ilə gəlmək olar?",
    "Restoranda tələbələr üçün endirim varmı?"
]

In [ ]:
for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 60}")
    print(f"TEST {i}")
    print("Sual:", question)

    answer = answer_question(question)

    print("Cavab:", answer)


TEST 1
Sual: Narxana neçə şəhərdə fəaliyyət göstərir?
Cavab: 

Salam! Narxana hazırda 6 şəhərdə fəaliyyət göstərir.

TEST 2
Sual: İçərişəhər filialında spirtli içki varmı?
Cavab: Bəli, İçərişəhər filialında **şərab kartı mövcuddur**. CONTEXT-də digər spirtli içkilər (votka, viski və s.) barədə ayrı məlumat verilməyib.

TEST 3
Sual: Restoranda rezervasiya necə edilir?
Cavab: Restoranda rezervasiya **3 üsulla** edilə bilər:  
- **Qısa nöqtəli xətt:** *1909*  
- **Mobil tətbiq**  
- **Filialın telefonu**

Rezervasiya **2–8 nəfərlik** masalar üçün **pulsuzdur**, amma **9 nəfərdən çox** qruplar və VIP otaqlar üçün **15 AZN/nəfər** depozit tələb olunur. Rezervasiya olunan masa **vaxtından sonra 20 dəqiqə** saxlanılır. Ləğv etmək üçün **ən azı 3 saat əvvəl** bildirmək lazımdır.

TEST 4
Sual: Çatdırılmada sifariş 90 dəqiqədən çox gecikərsə nə baş verir?
Cavab: User Safety: safe

TEST 5
Sual: Nar Xalı proqramında Qızıl səviyyənin cashback faizi neçədir?
Cavab: Nar Xalı proqramında Qızıl səviyy

In [ ]:
start_chat()

Narxana ağıllı ofisiant botuna xoş gəlmisiniz!
Menyu sualları üçün 'menu' sözündən istifadə edin.
Şəkil yaratmaq üçün 'generate' sözündən istifadə edin.
Çıxmaq üçün 'exit' və ya 'quit' yazın.



/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionXLPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionXLPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' obje

Siz: exit


/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionXLPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


Bot: Sağ olun! Yenidən gözləyirik.
